# **Data cleaning and feature selection**

## **Libraries**

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Make the local package importable from the project root or notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Start Jupyter from the project root or notebooks folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RAW_DATA_PATH,
    DESCRIPTION_DATA_PATH,
    CLEAN_DATA_PATH,
    CLEANING_RESULTS_DIR,
)
from src.preprocessing.eda import DatasetEDA
from src.preprocessing.cleaning import add_order_month, compare_dataset_quality
from src.preprocessing.descriptions import show_column_descriptions

# Configurations
pd.set_option("display.max_columns", None)


## **1. General EDA**

### **1.1 Data Loading and Preview**

In [2]:
df = pd.read_csv(RAW_DATA_PATH, encoding="latin-1")
eda = DatasetEDA(df, cache_dir=CLEANING_RESULTS_DIR)
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


### **1.2 Dataset Dimensions and Memory Usage**

In [3]:
eda.dimensions()

Loaded cached report: dimensions_899059cca98e8c4ee469a92c800b020303c32fd3c4e71eddb6efd8a847941583.json
Rows: 180,519
Columns: 53
Total cells: 9,567,507
Memory usage: 299.94 MiB


,Memory (MiB)
Product Image,20.477482
Product Name,14.481791
Customer Street,11.871404
Order State,11.323948
shipping date (DateOrders),10.932261
order date (DateOrders),10.931753
Delivery Status,10.914381
Order Country,10.900290
Shipping Mode,10.643388
Category Name,10.623387


### **1.3 Column Names and Data Types**

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

### **1.4 Missing Values**

In [5]:
missing_values = eda.missing_values()

Loaded cached report: missing_values_9d43e44c1f97b63250f33bc457fccfcf79bfca89aac40c01b0150617efe3d0c7.json


,Missing Count,Missing Percentage
Product Description,180519,100.00
Order Zipcode,155679,86.24
Customer Lname,8,0.00
Customer Zipcode,3,0.00
Days for shipment (scheduled),0,0.00
Sales per customer,0,0.00
Benefit per order,0,0.00
Delivery Status,0,0.00
Late_delivery_risk,0,0.00
Customer City,0,0.00


Total missing values: 336,209
Rows with missing values: 180,519


**Observations**
- Product description has null values in all the rows of the dataset.
- The order zip code has a high level of missing values (86.24%).
- The rest of the features don't have null values.

### **1.5 Duplicate Records**

In [6]:
duplicate_records = eda.duplicate_records()

Loaded cached report: duplicate_records_b5bb09fd063b37be0edb7b8e25636c3e7b28ed115477500f7e9cb1d35a3258b3.json
Duplicate rows: 0
Duplicate percentage: 0.00%


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode


**Observations**
- There aren't duplicated records.

In [7]:
identical_columns = eda.identical_columns()

Loaded cached report: identical_columns_11e43ed18c9658660a2ea57298c186a9641cbe9449fb5733b21a709364e57927.json
Pairs of identical columns: 7


,Column A,Column B,Identical Rows,Type A,Type B
0,Benefit per order,Order Profit Per Order,180519,float64,float64
1,Sales per customer,Order Item Total,180519,float64,float64
2,Category Id,Product Category Id,180519,int64,int64
3,Customer Email,Customer Password,180519,str,str
4,Customer Id,Order Customer Id,180519,int64,int64
5,Order Item Cardprod Id,Product Card Id,180519,int64,int64
6,Order Item Product Price,Product Price,180519,float64,float64


### **1.6 Unique Values and Constant Columns**

In [8]:
unique_values = eda.unique_values()

Loaded cached report: unique_values_5b9eaa738434d56d727fb80fe9a969fa359c2b0cf35f71b9c085b28531a8e03f.json


,Unique Count,Missing Count
Product Description,0,180519
Customer Password,1,0
Product Status,1,0
Customer Email,1,0
Late_delivery_risk,2,0
Customer Country,2,0
Customer Segment,3,0
Delivery Status,4,0
Days for shipment (scheduled),4,0
Shipping Mode,4,0


Constant columns: 3


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Customer Password,180519.0,1.0,XXXXXXXXX,180519.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product Status,180519.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Customer Email,180519.0,1.0,XXXXXXXXX,180519.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


All-null columns: 1
['Product Description']


**Observations**
- Customer Password, Product Status and Customer Email have 1 unique value.
- Order Item Id is a unique identifier.

### **1.7 Numerical Descriptive Statistics**

In [9]:
numerical_stats = eda.numerical_statistics()

Loaded cached report: numerical_statistics_36498f4a0ef6f808fad8006a52f67cfa3c81c1c7cbad527b3607d04491d8e6e0.json


,count,mean,std,min,1%,25%,50%,75%,99%,max,median,skewness,kurtosis
Days for shipping (real),180519.0,3.497654,1.623722,0.000000,0.000000,2.000000,3.000000,5.000000,6.000000,6.000000,3.000000,8.477127e-02,-1.007914
Days for shipment (scheduled),180519.0,2.931847,1.374449,0.000000,0.000000,2.000000,4.000000,4.000000,4.000000,4.000000,4.000000,-7.319983e-01,-1.022949
Benefit per order,180519.0,21.974989,104.433526,-4274.979980,-415.599402,7.000000,31.520000,64.800003,184.229996,911.799988,31.520000,-4.741834e+00,71.377259
Sales per customer,180519.0,183.107609,120.043670,7.490000,22.490000,104.379997,163.990005,247.399994,464.950012,1939.989990,163.990005,2.888446e+00,23.920362
Late_delivery_risk,180519.0,0.548291,0.497664,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-1.940742e-01,-1.962357
Category Id,180519.0,31.851451,15.640064,2.000000,7.000000,18.000000,29.000000,45.000000,74.000000,76.000000,29.000000,3.616248e-01,-0.603261
Customer Id,180519.0,6691.379495,4162.918106,1.000000,140.000000,3258.500000,6457.000000,9779.000000,18951.820000,20757.000000,6457.000000,4.887683e-01,0.014899
Customer Zipcode,180516.0,35921.126914,37542.461122,603.000000,725.000000,725.000000,19380.000000,78207.000000,97301.000000,99205.000000,19380.000000,4.908834e-01,-1.451419
Department Id,180519.0,5.443460,1.629246,2.000000,2.000000,4.000000,5.000000,7.000000,10.000000,12.000000,5.000000,2.733206e-01,-0.181697
Latitude,180519.0,29.719955,9.813646,-33.937553,18.201067,18.265432,33.144863,39.279617,45.359295,48.781933,33.144863,-9.796267e-02,-1.555415


**Observations**
- The feature Benefit per order, Order Profit Per Order and -2.750000 and Order Item Profit Ratio have negative values, but they don't really represent mistakes in the context.

### **1.8 Multiplicative Relationships**

In [10]:
multiplicative_dependencies = eda.multiplicative_relationships(
    relative_tolerance=1e-5,
    absolute_tolerance=0.01,
    min_valid_rows=30,
)

Loaded cached report: multiplicative_relationships_9f510273774ccc4b5dd8206127c4a7826028147280e12518a59733c4e8f160e2.json
Multiplicative relationships found: 2


,Column A,Column B,Target Column,Relationship,Rows Checked,Coverage (%),Maximum Absolute Error
0,Order Item Product Price,Order Item Quantity,Sales,[Order Item Product Price] * [Order Item Quantity] = [Sales],180519,100.0,0.000023
1,Order Item Quantity,Product Price,Sales,[Order Item Quantity] * [Product Price] = [Sales],180519,100.0,0.000023


**Observations**
- Sales presents a multiplicative relationship: Order Item Product Price (or Product Price) times Order Item Quantity.

# **2. Preprocessing**

We are going to delete the next columns:
- Product Description: all its values are null
- Order Id: its different for each record, it adds noise
- Order Zipcode: It has a high percentage of missing values.
- Customer Password, Product Status, Customer Email: They have just 1 unique value. Therefore, they add noise.
- Department Id: There exists in the dataset the department name.
- Order Item Product Price: It has the same information as Product Price
- Order Profit Per Order: It has the same information as Benefit per order.
- Order Item Total: It has the same information as Sales per customer
- Product Category Id: It has the same information as Category Id
- Order Item Cardprod Id: It has the same information as Product Card Id.
- order Item Id: Its a unique identifier, it adds noise.
- Category Id: we already have the Category Name.
- Order Customer Id: It has the same information as Customer Id. 
- Product Image: It doesn't represent usable information, because we aren't going to analyse images. Also, the links of the images don't work.
- Customer Street: we aren't going to analyse the streets of the customers.
- Customer Fname, Customer Lname: they don't represent relevant information for the study.
- shipping date (DateOrders),order date (DateOrders): we are only going to work with the month (order date) in the analysis to not make the models too complex.

In [11]:
df_before = df.copy(deep=True)
df = add_order_month(df)

drop_columns = [
    'Customer Email','Order Id','Product Description','Order Item Id',
    'Order Zipcode','Customer Password','Product Status',
    'Order Item Product Price','Department Id','Order Profit Per Order',
    'Order Item Total','Product Category Id','Order Item Cardprod Id',
    'Order Customer Id','Product Image','Category Id','Customer Fname','Customer Lname',
    'Customer Street','order date (DateOrders)','shipping date (DateOrders)']
df = df.drop(columns=drop_columns)

In [12]:
description_table = show_column_descriptions(
    dataframe=df,
    description_path=DESCRIPTION_DATA_PATH,
    cache_dir=CLEANING_RESULTS_DIR,
    additional_descriptions={
        "Order Month": "The month where the order was placed",
    },
)

Loaded cached report: show_column_descriptions_2919701097d5ace20669ab57ffd04ca4567f4f8f9765398cc21312e91da733ea.json


,Column,Description
0,Type,Type of transaction made
1,Days for shipping (real),Actual shipping days of the purchased product
2,Days for shipment (scheduled),Days of scheduled delivery of the purchased product
3,Benefit per order,Earnings per order placed
4,Sales per customer,Total sales per customer made per customer
5,Delivery Status,"Delivery status of orders: Advance shipping , Late delivery , Shipping canceled , Shipping on time"
6,Late_delivery_risk,"Categorical variable that indicates if sending is late (1), it is not late (0)."
7,Category Name,Description of the product category
8,Customer City,City where the customer made the purchase
9,Customer Country,Country where the customer made the purchase


### **2.1 Dataset Quality Before and After**

In [13]:
quality_comparison = compare_dataset_quality(
    before=df_before,
    after=df,
    cache_dir=CLEANING_RESULTS_DIR,
)

Loaded cached report: compare_dataset_quality_9df0a0924363350acf4fd56581369d7b9abdf6e10c0eedf83cd8548c33bf9a3f.json
Dataset quality: before and after


,Before,After,Difference (After - Before)
Metric,,,
Rows,180519.000000,180519.000000,0.000000
Columns,53.000000,33.000000,-20.000000
Missing cells,336209.000000,3.000000,-336206.000000
Missing cells (%),3.514071,0.000050,-3.514021
Rows with missing values,180519.000000,3.000000,-180516.000000
Duplicate rows after first,0.000000,0.000000,0.000000
Duplicate rows (%),0.000000,0.000000,0.000000
Constant columns (one non-null value),3.000000,0.000000,-3.000000
All-null columns,1.000000,0.000000,-1.000000


Column changes and quality indicators


,Status,Type Before,Missing Count Before,Missing (%) Before,Unique Count Before,Type After,Missing Count After,Missing (%) After,Unique Count After
Column,,,,,,,,,
Type,Retained,str,0.0,0.000000,4.0,str,0.0,0.000000,4.0
Days for shipping (real),Retained,int64,0.0,0.000000,7.0,int64,0.0,0.000000,7.0
Days for shipment (scheduled),Retained,int64,0.0,0.000000,4.0,int64,0.0,0.000000,4.0
Benefit per order,Retained,float64,0.0,0.000000,21998.0,float64,0.0,0.000000,21998.0
Sales per customer,Retained,float64,0.0,0.000000,2927.0,float64,0.0,0.000000,2927.0
Delivery Status,Retained,str,0.0,0.000000,4.0,str,0.0,0.000000,4.0
Late_delivery_risk,Retained,int64,0.0,0.000000,2.0,int64,0.0,0.000000,2.0
Category Id,Removed,int64,0.0,0.000000,51.0,NaN,NaN,NaN,NaN
Category Name,Retained,str,0.0,0.000000,50.0,str,0.0,0.000000,50.0


In [14]:
CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_DATA_PATH, index=False)